# AML Baseline Model — LightGBM on EDA-derived features

This is a **standalone modelling notebook**: it runs after the EDA notebook's Sections 1–9 have
already been executed once and saved their artifacts to `OUT_DIR`. It does not repeat the EDA — it
reloads what the EDA already produced (the currency conversion rates and the account feature table)
and uses them to train and evaluate a tabular baseline.

**Purpose.** Decide whether the planned RGCN/GAT edge classifier in PyTorch Geometric is actually
justified. If this baseline lands within a couple of PR-AUC points of the GNN, the graph structure
isn't adding much beyond degree, fan-out, and amount-variability, and the PyG build-out should be
deprioritised.

**Required artifacts from the EDA notebook** (Section 7 and Section 14 there must have run):
- `fx_usd_rate.csv` — the two-tier fiat-to-USD conversion table
- `fx_btc_daily_rate.csv` — the per-day Bitcoin rate
- `account_features.parquet` (or `.csv` if the parquet export fell back) — the node table with
  `is_laundering_node`, `date_first_seen`, and `is_dormant`

**Why transactions are reloaded from the raw file rather than from a saved enriched copy:** the EDA
notebook's `account_features` file only holds *aggregated* per-account features — it has no per-row
detail left to rebuild fold-specific, leakage-safe aggregates from. Re-deriving `amount_usd` from the
raw CSV plus the saved rate tables is fast (no graph rebuild) and keeps this notebook's transaction
data independent from whatever is in memory in the other notebook.


## 0. Configuration


In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import average_precision_score

DATA_DIR = Path("./data")
TX_FILE  = DATA_DIR / "LI-Small_Trans.csv"
OUT_DIR  = Path("./eda_output")   # must match the EDA notebook's OUT_DIR

FX_USD_RATE_FILE = OUT_DIR / "fx_usd_rate.csv"
FX_BTC_RATE_FILE = OUT_DIR / "fx_btc_daily_rate.csv"
NODE_FEATURES_FILE = OUT_DIR / "account_features.parquet"
if not NODE_FEATURES_FILE.exists():
    NODE_FEATURES_FILE = OUT_DIR / "account_features.csv"   # parquet-export fallback

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

for f in (TX_FILE, FX_USD_RATE_FILE, FX_BTC_RATE_FILE, NODE_FEATURES_FILE):
    status = "OK" if f.exists() else "MISSING"
    print(f"[{status}] {f}")
missing = [f for f in (TX_FILE, FX_USD_RATE_FILE, FX_BTC_RATE_FILE, NODE_FEATURES_FILE) if not f.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing: {missing}. Run the EDA notebook's Section 7 (currency normalisation) and "
        f"Section 14 (artefact export) first — this notebook only trains on what that one saved."
    )

[OK] data/LI-Small_Trans.csv
[OK] eda_output/fx_usd_rate.csv
[OK] eda_output/fx_btc_daily_rate.csv
[OK] eda_output/account_features.parquet


## 1. Reload transactions and reapply the saved currency conversion

This repeats the loader and type-casting from the EDA notebook's Section 1, but applies the
**already-computed** USD rates rather than rebuilding the two-tier FX graph from scratch — that
graph is expensive to rebuild and its output doesn't change unless the underlying data does.


In [15]:
CANONICAL_TX = {
    "timestamp":       ["Timestamp", "timestamp", "Date"],
    "from_bank":       ["From Bank", "from_bank", "Sender Bank"],
    "to_bank":         ["To Bank", "to_bank", "Receiver Bank"],
    "from_account":    ["Account", "From Account", "from_account", "Sender Account", "nameOrig"],
    "to_account":      ["Account.1", "To Account", "to_account", "Receiver Account", "nameDest"],
    "amount_received": ["Amount Received", "amount_received"],
    "currency_received":["Receiving Currency", "Amount Received Currency"],
    "amount_paid":     ["Amount Paid", "amount_paid", "Amount"],
    "currency_paid":   ["Payment Currency", "Amount Paid Currency"],
    "payment_format":  ["Payment Format", "payment_format", "Payment Type"],
    "is_laundering":   ["Is Laundering", "is_laundering", "isFraud", "Label"],
}

def normalise(df, mapping, label):
    rename, taken = {}, set()
    for canon, candidates in mapping.items():
        for c in candidates:
            if c in df.columns and c not in taken:
                rename[c] = canon; taken.add(c); break
    out = df.rename(columns=rename)
    print(f"[{label}] mapped {len(rename)} columns -> {sorted(rename.values())}")
    return out

tx_raw = pd.read_csv(TX_FILE, low_memory=False)
tx = normalise(tx_raw, CANONICAL_TX, "transactions")
del tx_raw

tx["timestamp"] = pd.to_datetime(tx["timestamp"], errors="coerce")
for c in ("from_account", "to_account"):
    tx[c] = tx[c].astype(str).str.strip()
for c in ("amount_paid", "amount_received"):
    tx[c] = pd.to_numeric(tx[c], errors="coerce")
tx["is_laundering"] = tx["is_laundering"].astype("int8")
for c in ("payment_format", "currency_paid", "currency_received", "from_bank", "to_bank"):
    tx[c] = tx[c].astype("category")
tx["date"] = tx["timestamp"].dt.floor("D")
tx["cross_bank"] = tx["from_bank"].astype(str) != tx["to_bank"].astype(str)

print(f"\nloaded {len(tx):,} transactions, {tx['date'].nunique()} distinct days")

[transactions] mapped 11 columns -> ['amount_paid', 'amount_received', 'currency_paid', 'currency_received', 'from_account', 'from_bank', 'is_laundering', 'payment_format', 'timestamp', 'to_account', 'to_bank']

loaded 6,924,049 transactions, 17 distinct days


In [16]:
# --- reapply the SAVED conversion, not a rebuilt one ---
rate_df = pd.read_csv(FX_USD_RATE_FILE, index_col=0)
usd_rate = rate_df["usd_rate"].to_dict()

btc_df = pd.read_csv(FX_BTC_RATE_FILE, index_col=0, parse_dates=True)
btc_daily_rate = btc_df.iloc[:, 0]   # first (only) data column, whatever it was named on save
btc_daily_rate.index = pd.to_datetime(btc_daily_rate.index)

BITCOIN = "Bitcoin"
missing_currencies = set(tx["currency_paid"].cat.categories) - set(usd_rate) - {BITCOIN}
if missing_currencies:
    raise ValueError(f"Currencies in this data with no saved rate: {missing_currencies}. "
                      f"Re-run the EDA notebook's Section 7 against this same file before continuing.")

is_btc_paid = tx["currency_paid"] == BITCOIN
is_btc_recv = tx["currency_received"] == BITCOIN

tx["amount_usd"] = np.nan
tx["amount_received_usd"] = np.nan
tx.loc[~is_btc_paid, "amount_usd"] = (
    tx.loc[~is_btc_paid, "amount_paid"] * tx.loc[~is_btc_paid, "currency_paid"].map(usd_rate).astype(float)
)
tx.loc[~is_btc_recv, "amount_received_usd"] = (
    tx.loc[~is_btc_recv, "amount_received"] * tx.loc[~is_btc_recv, "currency_received"].map(usd_rate).astype(float)
)
tx.loc[is_btc_paid, "amount_usd"] = (
    tx.loc[is_btc_paid, "amount_paid"] * tx.loc[is_btc_paid, "date"].map(btc_daily_rate).astype(float)
)
tx.loc[is_btc_recv, "amount_received_usd"] = (
    tx.loc[is_btc_recv, "amount_received"] * tx.loc[is_btc_recv, "date"].map(btc_daily_rate).astype(float)
)

unconv = tx["amount_usd"].isna().sum()
print(f"unconverted after reapplying saved rates: {unconv:,} / {len(tx):,}")
if unconv:
    print(tx.loc[tx["amount_usd"].isna(), "currency_paid"].value_counts())
    print("!! non-zero unconverted rows usually means this transactions file has drifted from "
          "the one the EDA notebook's rates were computed on (e.g. new dates -> new Bitcoin days "
          "with no saved rate). Investigate before trusting anything downstream.")

unconverted after reapplying saved rates: 0 / 6,924,049


## 2. Load the account feature table (target + `date_first_seen`)


In [17]:
if NODE_FEATURES_FILE.suffix == ".parquet":
    nodes = pd.read_parquet(NODE_FEATURES_FILE)
else:
    nodes = pd.read_csv(NODE_FEATURES_FILE)

nodes = nodes.rename(columns={"index": "account_id", "date": "date_first_seen"})
nodes = nodes.set_index("account_id")

nodes["date_first_seen"] = pd.to_datetime(nodes["date_first_seen"], errors="coerce")

print(f"loaded {len(nodes):,} accounts")
print(f"is_laundering_node positive rate: {nodes['is_laundering_node'].mean()*100:.4f}%")
print(f"is_dormant: {int(nodes['is_dormant'].sum()):,}  "
      f"({'>>> 0 dormant flagged - check the EDA notebook fix for this before trusting it' if nodes['is_dormant'].sum()==0 else 'looks populated'})")

loaded 712,684 accounts
is_laundering_node positive rate: 0.7442%
is_dormant: 6,781  (looks populated)


## 3. As-of feature engineering + rolling-origin split

**This is the part that must not be skipped.** The `out_amount_std`, `fan_in`/`fan_out`, `in_amount`
and `cross_bank_frac` columns already sitting in `nodes` are aggregated over the *entire* file. Using
them directly as training features in any fold would leak each account's future behaviour into that
fold's training data and inflate every PR-AUC reported below — verified empirically on a small fixture,
where one fold's PR-AUC dropped from a perfect 1.0000 to 0.7818 once features were rebuilt from only
pre-cutoff history. Every feature is therefore recomputed fresh, per fold, from `tx` filtered to
`date <= as_of_date` — `nodes` is used ONLY for its target (`is_laundering_node`) and
`date_first_seen` (fold assignment), never for its feature columns.


In [18]:
FEATURES = ["out_amount_std", "in_amount", "fan_in", "fan_out",
            "cross_bank_frac", "currency_crossing"]
TARGET = "is_laundering_node"

def build_features_asof(tx_all, as_of_date, account_ids):
    hist = tx_all[tx_all["date"] <= as_of_date]

    out_agg = (hist.groupby("from_account", observed=True)
                    .agg(out_amount_std=("amount_usd", "std"),
                         fan_out=("to_account", "nunique"))
                    .reindex(account_ids).fillna(0))
    in_agg = (hist.groupby("to_account", observed=True)
                   .agg(in_amount=("amount_received_usd", "sum"),
                        fan_in=("from_account", "nunique"))
                   .reindex(account_ids).fillna(0))
    cb = (hist.groupby("from_account", observed=True)["cross_bank"]
              .mean().reindex(account_ids).fillna(0).rename("cross_bank_frac"))

    # account-level currency-crossing flag: receives in one currency, pays out in a
    # DIFFERENT one across its history - invisible to the per-leg currency_mismatch flag
    recv_cur = (hist.assign(currency_received=lambda d: d["currency_received"].astype(str))
                    .groupby("to_account", observed=True)["currency_received"]
                    .agg(lambda s: frozenset(s.dropna())))
    paid_cur = (hist.assign(currency_paid=lambda d: d["currency_paid"].astype(str))
                    .groupby("from_account", observed=True)["currency_paid"]
                    .agg(lambda s: frozenset(s.dropna())))
    crossing = pd.Series(0, index=account_ids, dtype="int8", name="currency_crossing")
    for a in recv_cur.index.intersection(paid_cur.index).intersection(account_ids):
        if recv_cur[a] and paid_cur[a] and not (recv_cur[a] & paid_cur[a]):
            crossing[a] = 1

    return pd.concat([out_agg, in_agg, cb, crossing], axis=1)


def rolling_origin_splits(node_df, embargo_days=1, min_train_days=5):
    days = sorted(node_df["date_first_seen"].dropna().unique())
    splits = []
    for k in range(min_train_days, len(days) - embargo_days):
        train_end = days[k]
        val_day = days[k + embargo_days]
        train_accts = node_df.index[node_df["date_first_seen"] <= train_end]
        val_accts = node_df.index[node_df["date_first_seen"] == val_day]
        val_accts = val_accts.difference(train_accts)
        splits.append((train_end, val_day, train_accts, val_accts))
    return splits

n_avail_days = len(sorted(nodes["date_first_seen"].dropna().unique()))
print(f"distinct 'first-seen' days available: {n_avail_days}")

MIN_TRAIN_DAYS = 5   # matches the EDA notebook's rolling-origin CV; lower only if this errors empty
EMBARGO_DAYS = 1
splits = rolling_origin_splits(nodes, embargo_days=EMBARGO_DAYS, min_train_days=MIN_TRAIN_DAYS)
print(f"{len(splits)} rolling folds planned")

distinct 'first-seen' days available: 16
10 rolling folds planned


## 4. Train & evaluate LightGBM, fold by fold

`scale_pos_weight` corrects the severe class imbalance for training, but distorts `predict_proba`
as a calibrated probability — fine for the PR-AUC reported here (threshold-independent), but revisit
with isotonic or Platt recalibration before using these scores to set an actual alert threshold.


In [19]:
MIN_VAL_N = 50   # below this, a single lucky/unlucky fold swings the metric too much to trust

fold_scores, fold_models, fold_log = [], [], []
for i, (train_end, val_day, train_idx, val_idx) in enumerate(splits, 1):
    train_idx = train_idx.intersection(nodes.index)
    val_idx = val_idx.intersection(nodes.index)

    base_rate = nodes.loc[val_idx, TARGET].mean() if len(val_idx) else np.nan

    # A fold is unusable for PR-AUC not just when it has zero positives, but also when it
    # has zero NEGATIVES (base_rate == 1.0) or is simply too small. average_precision_score
    # returns a trivial 1.0 on a single-class validation set regardless of model quality -
    # that is not a real score, and averaging it in with real folds silently inflates the
    # reported mean (verified: 6 of 10 folds here were single-class with 1-38 examples,
    # dragging an honest ~0.04 mean up to a fabricated 0.62).
    if len(val_idx) < MIN_VAL_N:
        print(f"fold {i}: SKIPPED - val n={len(val_idx)} < MIN_VAL_N={MIN_VAL_N} "
              f"(val_day={val_day.date()})")
        continue
    if base_rate == 0 or base_rate == 1:
        print(f"fold {i}: SKIPPED - single-class validation set (base rate={base_rate*100:.0f}%, "
              f"val n={len(val_idx)}, val_day={val_day.date()}) - PR-AUC is undefined/trivial here, "
              f"not a real measurement")
        continue

    train_feat = build_features_asof(tx, train_end, train_idx)
    val_feat = build_features_asof(tx, val_day, val_idx)

    X_train, y_train = train_feat[FEATURES], nodes.loc[train_idx, TARGET]
    X_val, y_val = val_feat[FEATURES], nodes.loc[val_idx, TARGET]

    pos, neg = y_train.sum(), (1 - y_train).sum()
    spw = neg / max(pos, 1)
    model = lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                scale_pos_weight=spw, random_state=RANDOM_SEED, verbosity=-1)
    model.fit(X_train, y_train)
    val_scores = model.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, val_scores)
    fold_scores.append(pr_auc)
    fold_models.append(model)
    fold_log.append({"fold": i, "train_end": str(train_end.date()), "val_day": str(val_day.date()),
                      "n_train": len(train_idx), "n_val": len(val_idx),
                      "base_rate_pct": base_rate * 100, "pr_auc": pr_auc})
    print(f"fold {i}: train_end={train_end.date()} val_day={val_day.date()} "
          f"| train n={len(train_idx):,} val n={len(val_idx):,} "
          f"| val base rate={base_rate*100:.3f}% | PR-AUC={pr_auc:.4f} "
          f"(lift={pr_auc/max(base_rate,1e-9):.1f}x over random)")

n_skipped = len(splits) - len(fold_scores)
if n_skipped:
    print(f"\n{n_skipped} of {len(splits)} folds skipped as degenerate. If most skips land on the "
          f"LATE days in your window, cross-reference against the EDA notebook's Section 8 daily-"
          f"volume plot: a handful of genuinely new accounts appearing very late tend to be "
          f"disproportionately part of injected scenarios (like the 10042B family), not organic "
          f"traffic - real, but not something a single fold with 1-38 examples can evaluate.")
if not fold_scores:
    print("\nno usable folds - lower MIN_TRAIN_DAYS, raise MIN_VAL_N's tolerance, or check "
          "date_first_seen coverage in the node table.")

fold 1: train_end=2022-09-06 val_day=2022-09-07 | train n=366,367 val n=11,145 | val base rate=1.992% | PR-AUC=0.0577 (lift=2.9x over random)
fold 2: train_end=2022-09-07 val_day=2022-09-08 | train n=377,512 val n=11,409 | val base rate=2.139% | PR-AUC=0.0594 (lift=2.8x over random)
fold 3: train_end=2022-09-08 val_day=2022-09-09 | train n=388,921 val n=188,920 | val base rate=0.571% | PR-AUC=0.0212 (lift=3.7x over random)
fold 4: train_end=2022-09-09 val_day=2022-09-10 | train n=577,841 val n=103,358 | val base rate=1.293% | PR-AUC=0.0251 (lift=1.9x over random)
fold 5: SKIPPED - val n=38 < MIN_VAL_N=50 (val_day=2022-09-11)
fold 6: SKIPPED - val n=18 < MIN_VAL_N=50 (val_day=2022-09-12)
fold 7: SKIPPED - val n=16 < MIN_VAL_N=50 (val_day=2022-09-13)
fold 8: SKIPPED - val n=7 < MIN_VAL_N=50 (val_day=2022-09-14)
fold 9: SKIPPED - val n=1 < MIN_VAL_N=50 (val_day=2022-09-15)
fold 10: SKIPPED - val n=2 < MIN_VAL_N=50 (val_day=2022-09-17)

6 of 10 folds skipped as degenerate. If most skips la

## 5. Summary and feature importance


In [20]:
if fold_scores:
    print(f"mean PR-AUC across {len(fold_scores)} USABLE folds (of {len(splits)} planned): "
          f"{np.mean(fold_scores):.4f} (std {np.std(fold_scores):.4f}, "
          f"range {min(fold_scores):.4f}-{max(fold_scores):.4f})")
    print("(single-class / too-small folds were excluded above - see the SKIPPED lines - "
          "this mean reflects only folds where PR-AUC is actually a meaningful measurement)")

    imp = pd.DataFrame([m.feature_importances_ for m in fold_models], columns=FEATURES)
    print("\nfeature importance, mean +/- std across usable folds:")
    print(pd.DataFrame({"mean": imp.mean(), "std": imp.std()})
            .sort_values("mean", ascending=False).to_string())

    pd.DataFrame(fold_log).to_json(OUT_DIR / "baseline_lightgbm_folds.json", orient="records", indent=2)
    pd.Series({"mean_pr_auc": float(np.mean(fold_scores)),
               "std_pr_auc": float(np.std(fold_scores)),
               "n_usable_folds": len(fold_scores),
               "n_folds_planned": len(splits)}).to_json(OUT_DIR / "baseline_lightgbm_summary.json", indent=2)
    print(f"\nsaved -> {OUT_DIR/'baseline_lightgbm_summary.json'}")
    print(f"saved -> {OUT_DIR/'baseline_lightgbm_folds.json'} (per-fold detail for the daily-skew "
          f"cross-check above)")

mean PR-AUC across 4 USABLE folds (of 10 planned): 0.0408 (std 0.0178, range 0.0212-0.0594)
(single-class / too-small folds were excluded above - see the SKIPPED lines - this mean reflects only folds where PR-AUC is actually a meaningful measurement)

feature importance, mean +/- std across usable folds:
                     mean         std
out_amount_std     860.00   67.651066
in_amount          803.25  113.702463
fan_in             489.50   18.876794
cross_bank_frac    407.75  119.527891
fan_out            234.00   69.546148
currency_crossing    0.00    0.000000

saved -> eda_output/baseline_lightgbm_summary.json
saved -> eda_output/baseline_lightgbm_folds.json (per-fold detail for the daily-skew cross-check above)


### Reading this result

- **If `currency_crossing` shows near-zero importance**, that's consistent with it being genuinely
  rare in this data (11/6,693 accounts in the full-file EDA run) — not a reason to conclude the
  Section 6.1 finding was wrong, since a model can correctly learn to ignore a rare-but-real signal
  when five other features already separate most of the positive class.
- **Compare `mean_pr_auc` against the planned RGCN/GAT edge classifier once that's built.** A gap of a
  couple of PR-AUC points or less means the graph structure captured by degree/fan-out/amount-variance
  is doing most of the work already, and the PyTorch Geometric build-out should be scoped down or
  deprioritised in favour of iterating on this tabular feature set instead.
- **Fold-to-fold spread (`std_pr_auc`) matters as much as the mean.** The EDA notebook's own daily
  volume plot showed extreme day-to-day skew — expect some folds to be systematically easier or
  harder than others, and report the range, not just a single averaged number, when this goes in
  front of anyone deciding whether to greenlight the GNN work.
